## Baselines: what every model has to beat

Before comparing the models, we need reference points based on the same folds and with the same quota:
- **Random selection**: the floor. If a model does not beat this, it has learned nothing useful.
- **Government formula** (`pcexppred`): the real-world benchmark, the figure to beat.
- **Perfect ranking**: the ceiling. We rank based on true consumption, which no model can know. It’s not 100% because, in many villages, there are more poor households than there are transfers. 
- **Government refit**: the same type of formula (OLS on government variables), estimated again on our training villages. 

In [1]:
import sys
sys.path.append("..")   # so the notebook finds the shared code in src/ which is one folder up

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import PredefinedSplit

# shared group code with the same seed, same data, same evaluation
from src.utils import SEED, load_data, select_by_quota, recall_at_quota, within_village_auc
from src.features import SET_A, DISTRICT, make_preprocessor

train, test = load_data()    # clean data, so the training villages with fold and the test villages
# test is not used here because test villages are only used once at the very end for final evaluation                  
cv = PredefinedSplit(train["fold"])   #  the 5 saved folds of the villages
# the saved folds are not a new split, because all models need to be compared on the same villages         
y = train["LOGCONSUMPTION"] # the target of the refit in log consumption per person
print("train:", train.shape, "| villages:", train["hhea"].nunique(), "| SEED:", SEED)

train: (4090, 113) | villages: 512 | SEED: 40


### Government refit: with and without district dummies
The government estimates one formula per district and area. We fit **one pooled OLS** instead, because there are too few households in some of the areas. It is thus a *simplified version* of the government's method.

In [2]:

# the two versions of the refit:
# - "with" the district dummies, so close to the government
# - "without" because a district is the same for every household in a village so it cannot change who is chosen inside a village
refits = {
    "Refit with districts":    SET_A,                                        
    "Refit without districts": [c for c in SET_A if c not in DISTRICT],   # same list but the district is removed   
}

def make_X(cols):
    X = train[cols].copy() # copy, so the original table is not changed
    num_cols = [c for c in cols if c not in DISTRICT]
    X[num_cols] = X[num_cols].astype(float)
    # float because <NA> becomes NaN and the model cannot handle <NA> values, but it can handle NaN values
    # not the district  because it is not a number
    return X

### All baselines, fold by fold

For each of the 5 folds, we use the validation villages of that fold and:
1. **Random**: give every household a random number and select by quota. One draw is luck, so we repeat it 100 times and take the average.
2. **Government formula** and **perfect ranking**: nothing to fit, we only rank by `pcexppred` or by the true consumption.
3. **Refits**: learn the OLS on the 4 other folds, then predict this fold. The model never sees the villages it is evaluated on.

For every model we compute the **recall at the quota** (main metric) and the **within-village AUC** (share of poor/non-poor pairs in the same village ranked correctly).

In [3]:
rng = np.random.default_rng(SEED)
rows = []
for k, (train_idx, val_idx) in enumerate(cv.split()):   # k = fold number (0 to 4)
    val = train.iloc[val_idx].copy()                         

    # 1. Random selection (the floor)
    random_recalls = []
    for i in range(100):    # 100 draws, because one draw alone depends on luck
        val["random_score"] = rng.random(len(val))  # a random number for every household
        random_recalls.append(recall_at_quota(val, select_by_quota(val, "random_score", higher_is_poorer=True)))
    rows.append({"model": "Random", "fold": k, "recall": np.mean(random_recalls), "auc": np.nan})

    # 2. Government formula and 3. perfect ranking (no fitting, only ranking)
    for name, score in [("Government formula", "pcexppred"), ("Perfect ranking", "LOGCONSUMPTION")]:
        selected = select_by_quota(val, score, higher_is_poorer=False)
        rows.append({"model": name, "fold": k, "recall": recall_at_quota(val, selected),
                     "auc": within_village_auc(val, score, higher_is_poorer=False)})

    # 4. Government refits: fit on the 4 other folds, predict fold k
    for name, cols in refits.items():
        X = make_X(cols)
        model = Pipeline([("prep", make_preprocessor(cols, drop_first=True)),   
                          ("model", LinearRegression())])   # OLS, like the government
        # drop_first=True because OLS without penalty needs one district left out as reference
        # in a Pipeline because the medians and scaling are learned on the training folds only, so no leakage
        model.fit(X.iloc[train_idx], y.iloc[train_idx])     # learn on the 4 other folds
        val["pred"] = model.predict(X.iloc[val_idx])    # predicted log consumption in fold k
        selected = select_by_quota(val, "pred", higher_is_poorer=False)
        rows.append({"model": name, "fold": k, "recall": recall_at_quota(val, selected),
                     "auc": within_village_auc(val, "pred", higher_is_poorer=False)})

fold_results = pd.DataFrame(rows)
display(fold_results.pivot(index="fold", columns="model", values="recall").round(3))    # recall to have one row per fold 

model,Government formula,Perfect ranking,Random,Refit with districts,Refit without districts
fold,,,,,
0,0.559,0.716,0.370,0.585,0.595
1,0.544,0.682,0.374,0.585,0.582
2,0.548,0.664,0.353,0.535,0.551
3,0.552,0.729,0.382,0.576,0.569
4,0.498,0.639,0.341,0.520,0.520


In [4]:
# Report of the mean and the standard deviation of the models across the 5 folds
summary = fold_results.groupby("model")[["recall", "auc"]].agg(["mean", "std"])
summary = summary.sort_values(("recall", "mean"))
display(summary.round(3))

# share of the possible improvement: where each model lies between the floor and the ceiling
r_random = summary.loc["Random", ("recall", "mean")]
r_perfect = summary.loc["Perfect ranking", ("recall", "mean")]
for name in ["Government formula", "Refit with districts", "Refit without districts"]:
    share = (summary.loc[name, ("recall", "mean")] - r_random) / (r_perfect - r_random)
    print(f"{name}: {share:.0%} of the possible improvement")

recall           auc       
                          mean    std   mean    std
model                                              
Random                   0.364  0.017    NaN    NaN
Government formula       0.540  0.024  0.777  0.023
Refit with districts     0.560  0.031  0.825  0.021
Refit without districts  0.564  0.029  0.826  0.023
Perfect ranking          0.686  0.037  1.000  0.000

Government formula: 55% of the possible improvement
Refit with districts: 61% of the possible improvement
Refit without districts: 62% of the possible improvement


**Results:**

- **Random selection** finds about 36 out of 100 poor households. This is the floor: any useful model has to do better.
- **The government formula** finds 54 out of 100. It uses 55% of the possible improvement between random and perfect.
- **The government refit** (same kind of formula, estimated again on our training villages) finds about 56 out of 100, about 2 more than the government. Its within-village AUC is also higher (0.83 vs. 0.78), so it ranks neighbours better.
- **Districts do not help**: the refit with districts (56.0%) and without (56.4%) are the same. A district is identical for every household in a village, so it cannot change who is chosen inside a village.
- **The perfect ranking** finds only 69 out of 100, not 100. In many villages there are more poor households than transfers, so even a perfect model cannot help all of them. The realistic room to improve is about 32 points (from 36% to 69%).
- **But the gain over the government is small**: 2 points, while the results move by about 3 points from one fold to the next (std). So we cannot say yet that the refit is really better. The test set will show if it holds in new villages as well.